In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import absl.logging
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableSequence, RunnableParallel
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI

In [3]:
!unzip "/content/drive/MyDrive/full_contracts/CUAD_v1.zip"

Archive:  /content/drive/MyDrive/full_contracts/CUAD_v1.zip
replace CUAD_v1/CUAD_v1.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: CUAD_v1/CUAD_v1.json    
replace CUAD_v1/CUAD_v1_README.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: ALL
  inflating: CUAD_v1/CUAD_v1_README.txt  
  inflating: CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement.pdf  
  inflating: CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf  
  inflating: CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.pdf  
  inflating: CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.pdf  
  inflating: CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/SouthernStarEnergyInc_20051202_SB-2A_EX-9

In [4]:
!pip install "unstructured[pdf]"

In [ ]:
from langchain_community.document_loaders.directory import DirectoryLoader

#Loading all the PDF files from multiple directories into memory
paths = [r"/content/CUAD_v1/full_contract_pdf/Part_I",
     r"/content/CUAD_v1/full_contract_pdf/Part_II",
     r"/content/CUAD_v1/full_contract_pdf/Part_III"]

docs = []
for path in paths:
  loader = DirectoryLoader(path=path,
                           glob="**/*.pdf",
                           show_progress=True
  )
  docs.extend(loader.load())

len(docs)

  0%|          | 0/181 [00:00<?, ?it/s]

  1%|          | 2/181 [00:07<10:31,  3.53s/it]

  2%|▏         | 3/181 [00:07<06:26,  2.17s/it]

  2%|▏         | 4/181 [00:08<04:48,  1.63s/it]

In [ ]:
docs[10]

# Chucking
spliting the documents into smaller chucks for embedding

In [ ]:
#chucking

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitters=RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

text_chucking=chucks = text_splitters.split_documents(docs)



In [ ]:
text_chucking


In [ ]:
text_chucks = []
for doc in text_chucking:
    text_chucks.append(doc.page_content)

In [ ]:
text_chucks

#Embedding
Will using Sentence Trasnformer model into embedding

In [ ]:
!pip install -U sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer
# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
#downloading the model to aviod SSL error
!pip install "huggingface_hub[hf_xet]"


In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:

# Create embeddings using the sentence-transformer model
#embeddings_array = model.encode(text_chucks)

#Vector Store

In [ ]:
!pip install chromadb

In [ ]:
embeddings = model.encode(text_chucks)

# Create the Chroma vector store
vector_db = Chroma(
    embedding_function=model.encode,
    persist_directory="chroma_db",
    collection_name="sample"
)

vector_db.add_texts(text_chucks, embeddings=embeddings)